In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem, DataStructs
from rdkit.Avalon import pyAvalonTools
from modules.data_loader import load_cleaned, save_features

In [2]:
visco = load_cleaned()
visco.info()

<class 'pandas.DataFrame'>
RangeIndex: 8152 entries, 0 to 8151
Data columns (total 21 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_surfactant1      8152 non-null   str    
 1   type_surfactant1    8152 non-null   str    
 2   smiles_surfactant1  8152 non-null   str    
 3   c_surfactant1       8152 non-null   float64
 4   id_surfactant2      2726 non-null   str    
 5   type_surfactant2    2726 non-null   str    
 6   smiles_surfactant2  2631 non-null   str    
 7   c_surfactant2       8152 non-null   float64
 8   id_additive1        5504 non-null   str    
 9   smiles_additive1    5504 non-null   str    
 10  c_additive1         8152 non-null   float64
 11  id_additive2        1295 non-null   str    
 12  smiles_additive2    1139 non-null   str    
 13  c_additive2         8152 non-null   float64
 14  temperature         8152 non-null   int64  
 15  viscoelasticity     8152 non-null   float64
 16  g_0              

In [3]:
smiles = visco['smiles_surfactant1']
smiles = pd.concat([smiles, visco['smiles_surfactant2']])
smiles = pd.concat([smiles, visco['smiles_additive1']])
smiles = pd.concat([smiles, visco['smiles_additive2']])
smiles = smiles.drop_duplicates()
smiles = smiles.dropna()
len(smiles)

137

In [4]:
smiles = smiles.to_frame(name="smiles")

In [5]:
def smiles_to_avalon(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    fp = pyAvalonTools.GetAvalonFP(mol, nBits=256)
    arr = np.zeros(fp.GetNumBits())
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

In [6]:
avalon = pd.DataFrame(
    smiles["smiles"].apply(smiles_to_avalon).tolist(),
    index=smiles.index,
    columns=[f"avalon_{i}" for i in range(0, 256)]
)

avalon = pd.concat([smiles, avalon], axis=1)

In [7]:
avalon.info()

<class 'pandas.DataFrame'>
Index: 137 entries, 0 to 4640
Columns: 257 entries, smiles to avalon_255
dtypes: float64(256), str(1)
memory usage: 276.1 KB


In [9]:
save_features(avalon, 'avalon')